# ResNet-50 CNN — standard image pipeline, 224 px (R4)

This notebook fine-tunes an **ImageNet-pretrained ResNet-50** to classify the 200 bird species, using the team's image data pipeline **exactly as [`05_image_preparation.ipynb`](05_image_preparation.ipynb) describes and checks it** (*Decision 9*):

| Step | Where it comes from |
|---|---|
| Check every photo, write `data/cub_images.csv` | [`imagePrep.ipynb`](imagePrep.ipynb) |
| RGB, crop to the bounding box + 10% margin, shorter side 256 px | `scripts/prepare_images.py` (default settings) |
| Pipeline checks: 11,788 RGB photos at 256 px, no train / validation / test overlap | `05_image_preparation.ipynb`, **run by this notebook** (section 4); training only starts if every check passes |
| Train / validation / test split (validation = 20% of the official training photos, stratified by species) | `make_splits` in `scripts/augmentation.py` |
| Training augmentation (random crop to 224 px, flip, ±15° rotation, lighting jitter, random erasing) and the fixed validation / test transform (resize 256, centre crop 224) | `train_transform` / `eval_transform` in `scripts/augmentation.py`, imported unchanged |

Nothing about the photos is changed here, so the model input is **224 px**. [`resnet50.ipynb`](resnet50.ipynb) is the variant that departs from this pipeline (512 px photos, 448 px input) and scored 0.818 macro-F1; this notebook shows what the team's standard pipeline gives.

**Main metric:** macro-F1 on the official test split (accuracy and top-5 accuracy alongside). The report should say that the photos are cropped with the dataset's bounding boxes on train **and** test.

### Protocol (no test leakage)
1. **Select** (section 7): for each backbone learning rate, train on the 80% training part and score the validation part after every epoch. The rate with the best validation macro-F1 wins.
2. **Final** (section 8): retrain from the ImageNet weights on all 5,994 training photos with that rate, for the same number of epochs, and keep the last epoch.
3. **Test** (section 9): the official test photos are scored **once**, at the end.

### How to run on Colab
1. Open this notebook from GitHub in Colab (File → Open notebook → GitHub → `Flamingsides/f20dl-coursework-1`, branch `Ryan`).
2. Runtime → Change runtime type → **L4 GPU** (a T4 also works, about 2× slower).
3. The first time only: put your `kaggle.json` in Google Drive at `MyDrive/f20dl_resnet50/kaggle.json`.
4. Runtime → Run all. Every session downloads the original photos (~2 min; `05` needs them for its before / after figure). The cropped photos are made once (~10 min) and cached in Drive as `cub_processed_256px.zip`, the same cache the `test` profile of `resnet50.ipynb` uses.

If Colab disconnects, run all again: every run resumes from its last epoch saved in Drive, and finished runs are skipped.

## 1. Settings

In [ ]:
SMOKE_TEST = False     # True = tiny subset, 1 epoch: only checks that the code runs end to end

REPO_URL = "https://github.com/Flamingsides/f20dl-coursework-1.git"
BRANCH = "Ryan"
DRIVE_FOLDER = "f20dl_resnet50"   # folder in MyDrive for the data cache, checkpoints and results

BATCH_SIZE = 64
EPOCHS = 25
# Backbone learning rates to try. resnet50.ipynb's full run chose 0.003 at batch 32 (the top of its grid);
# at batch 64 the same step per photo is 0.006, so try that, half of it and double it.
LR_GRID = [0.003, 0.006, 0.012]
WARMUP_EPOCHS, WEIGHT_DECAY = 1, 1e-4
HEAD_LR_MULT = 10            # the new 200-way classifier starts from random, so it learns faster
LABEL_SMOOTHING = 0.1
WEIGHTS = "IMAGENET1K_V1"    # the standard ImageNet ResNet-50 weights used in most CUB papers

if SMOKE_TEST:
    EPOCHS, WARMUP_EPOCHS, BATCH_SIZE, LR_GRID = 1, 0, 8, LR_GRID[:2]

BASE_NAME = "resnet50_standard_224px" + ("_smoke" if SMOKE_TEST else "")
CONFIG = dict(pipeline="05_image_preparation", weights=WEIGHTS, img_size=224, store_px=256,
              batch_size=BATCH_SIZE, epochs=EPOCHS, warmup_epochs=WARMUP_EPOCHS, head_lr_mult=HEAD_LR_MULT,
              weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTHING, smoke_test=SMOKE_TEST)
print(f"{BASE_NAME}: backbone learning rates {LR_GRID}")
CONFIG

## 2. Environment

On Colab: mount Google Drive and clone the repository (for the pipeline notebooks and scripts). Locally: use this repository.

In [ ]:
import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from IPython.display import Image as ShowImage, display

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE.mkdir(parents=True, exist_ok=True)
    ROOT = Path("/content/f20dl-coursework-1")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
    OUT = DRIVE                                    # survives disconnects
    CKPT_DIR = DRIVE / "checkpoints"
else:
    DRIVE = None
    ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
    OUT = ROOT / "results" / "resnet50"
    CKPT_DIR = ROOT / "data" / "processed" / "checkpoints"   # git-ignored: checkpoints are ~100 MB

FIGURES = OUT / "figures"
for d in (OUT, CKPT_DIR, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "scripts"))

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True          # picks the fastest convolution algorithms for fixed-size inputs
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"                    # mixed precision: ~2x faster, same accuracy
GPU = torch.cuda.get_device_name() if DEVICE.type == "cuda" else ""

print(f"Colab: {IN_COLAB} | device: {DEVICE} {GPU} | {os.cpu_count()} CPU cores")
print(f"torch {torch.__version__} | outputs -> {OUT}")
if IN_COLAB and DEVICE.type != "cuda":
    raise RuntimeError("No GPU: Runtime -> Change runtime type -> L4 GPU, then run all again.")

## 3. Build the cropped photos (imagePrep → prepare_images.py)

1. **Original photos**: downloaded from Kaggle with `scripts/download_data.py` if missing.
2. **Cropped photos**: restored from the Drive cache if it exists; otherwise `imagePrep.ipynb` checks every photo and `scripts/prepare_images.py` (default settings: 256 px) crops them, and the result is cached in Drive.

Colab does not show the output of these helper programs while they run, so this cell can look idle for a few minutes (up to ~10 the first time).

In [ ]:
DATA = ROOT / "data"
PROCESSED = DATA / "processed"
CSV = PROCESSED / "cub_images_cropped.csv"         # the file 05_image_preparation.ipynb checks
IMG_DIR = PROCESSED / "images_cropped"
CACHE_ZIP = DRIVE / "cub_processed_256px.zip" if IN_COLAB else None

def run(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True, cwd=ROOT)

# 1. Original photos (imagePrep needs them, and so does the before / after figure in 05)
if not (any((DATA / "raw").rglob("image_class_labels.txt")) or (ROOT / "CUB_200_2011").exists()):
    if IN_COLAB:
        token = DRIVE / "kaggle.json"
        if not token.exists():
            raise FileNotFoundError(f"Put your kaggle.json in Google Drive at {token} (see the top of this notebook)")
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(exist_ok=True)
        shutil.copy(token, kaggle_dir / "kaggle.json")
        os.chmod(kaggle_dir / "kaggle.json", 0o600)
    run([sys.executable, "scripts/download_data.py"])

# 2. Cropped photos
if not CSV.exists() and IN_COLAB and CACHE_ZIP.exists():
    print("Restoring the cropped photos from the Drive cache...", flush=True)
    shutil.unpack_archive(CACHE_ZIP, DATA)
if not CSV.exists():
    if not (DATA / "cub_images.csv").exists():
        run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", "--output-dir", PROCESSED / "_executed",
             "notebooks/imagePrep.ipynb"])
    run([sys.executable, "scripts/prepare_images.py"])
    if IN_COLAB:
        print("Caching the cropped photos in Drive...", flush=True)
        local_zip = Path("/content") / CACHE_ZIP.name     # build locally, then one copy to Drive (much faster)
        with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_STORED) as z:   # JPEGs are already compressed
            z.write(CSV, CSV.relative_to(DATA))
            for p in IMG_DIR.rglob("*"):
                if p.is_file():
                    z.write(p, p.relative_to(DATA))
        shutil.copy(local_zip, CACHE_ZIP)

print(f"{CSV.relative_to(ROOT)}: {len(pd.read_csv(CSV)):,} photos")

## 4. Run `05_image_preparation.ipynb`

The pipeline notebook is executed as it is in the repository. Its checks stop this notebook with an error if anything is wrong:
- all 11,788 photos were cropped, are RGB and have a 256 px shorter side;
- train, validation and test share no photo, validation comes from the official training split, and all 200 species are in train and validation.

Its executed copy is saved next to the results, and its two figures are shown below.

In [ ]:
EXECUTED = PROCESSED / "_executed"
run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", "--output-dir", EXECUTED,
     "notebooks/05_image_preparation.ipynb"])

executed = json.loads((EXECUTED / "05_image_preparation.ipynb").read_text())
for cell in executed["cells"]:
    for out in cell.get("outputs", []):
        if out.get("name") == "stdout":
            print("".join(out["text"]).strip())
if IN_COLAB:
    shutil.copy(EXECUTED / "05_image_preparation.ipynb", OUT / "05_image_preparation_executed.ipynb")
for name in ("05_crop_examples", "05_augmentation_examples"):
    display(ShowImage(filename=str(ROOT / "documentation" / "figures" / f"{name}.png"), width=800))

## 5. Splits, transforms and loaders from `scripts/augmentation.py`

The same `make_splits`, `train_transform`, `eval_transform` and `BirdImages` that `05` checks and shows, imported unchanged. Only the loader settings (batch size, worker processes) are chosen here, for speed on the GPU.

In [ ]:
from augmentation import (BirdImages, make_splits, train_transform, eval_transform,
                          IMG_SIZE, N_CLASSES, RANDOM_STATE)

assert IMG_SIZE == CONFIG["img_size"]
train_df, val_df, test_df = make_splits(CSV)
if SMOKE_TEST:   # 10 species, a few photos each
    pick = lambda df, n: df[df.class_id <= 10].groupby("class_id").head(n)
    train_df, val_df, test_df = pick(train_df, 4), pick(val_df, 2), pick(test_df, 3)
full_train_df = pd.concat([train_df, val_df])     # all official training photos, for the final retrain
assert not set(full_train_df.image_id) & set(test_df.image_id)
assert not set(train_df.image_id) & set(val_df.image_id)

NUM_WORKERS = 0 if SMOKE_TEST else min(12, os.cpu_count() or 2)

def make_loader(df, train):
    return DataLoader(BirdImages(df, PROCESSED, train_transform if train else eval_transform),
                      batch_size=BATCH_SIZE, shuffle=train, drop_last=train, num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type == "cuda", persistent_workers=NUM_WORKERS > 0,
                      generator=torch.Generator().manual_seed(RANDOM_STATE) if train else None)

species = (pd.read_csv(CSV)[["class_id", "species"]].drop_duplicates()
           .set_index("class_id").sort_index().species.tolist())       # index = class_id - 1
print(f"train {len(train_df):,} | val {len(val_df):,} | all training {len(full_train_df):,} | "
      f"test {len(test_df):,} photos | input {IMG_SIZE} px | {NUM_WORKERS} loader workers")

## 6. Model and training code

- **ResNet-50** with its 1000-class ImageNet classifier replaced by a new 200-class layer.
- **Two learning rates:** the pretrained layers move 10× more slowly than the new classifier, so fine-tuning adjusts the ImageNet features instead of overwriting them.
- **SGD with momentum 0.9**, one warm-up epoch, then cosine decay to zero (updated every batch).
- **Label smoothing 0.1:** the target is 90% on the true species and the rest spread over the others, which discourages over-confident memorising of ~30 photos per species.
- **Mixed precision** and channels-last memory layout for speed on the GPU.
- Each run saves a checkpoint to Drive after every epoch and resumes from it after a disconnect.

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

def new_model():
    torch.manual_seed(RANDOM_STATE)
    model = resnet50(weights=ResNet50_Weights[WEIGHTS], progress=False)
    model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
    return model.to(DEVICE, memory_format=torch.channels_last)

def new_optimiser(model, lr_backbone, steps_per_epoch):
    head = list(model.fc.parameters())
    head_ids = {id(p) for p in head}
    backbone = [p for p in model.parameters() if id(p) not in head_ids]
    optimizer = torch.optim.SGD([{"params": backbone, "lr": lr_backbone},
                                 {"params": head, "lr": lr_backbone * HEAD_LR_MULT}],
                                momentum=0.9, nesterov=True, weight_decay=WEIGHT_DECAY)
    steps, warmup = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch
    def lr_factor(step):
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, steps - warmup)))
    return optimizer, torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)

def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    loss_sum, correct, seen = 0.0, 0, 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(x)
            loss = criterion(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        loss_sum += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        seen += len(y)
    return loss_sum / seen, correct / seen

@torch.no_grad()
def predict(model, loader):
    """Class probabilities, true labels (0-199) and mean cross-entropy (without smoothing)."""
    model.eval()
    probs, labels, loss_sum = [], [], 0.0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(x)
        loss_sum += nn.functional.cross_entropy(logits.float(), y, reduction="sum").item()
        probs.append(logits.float().softmax(1).cpu())
        labels.append(y.cpu())
    probs, labels = torch.cat(probs).numpy(), torch.cat(labels).numpy()
    return probs, labels, loss_sum / len(labels)

def scores(probs, labels):
    preds = probs.argmax(1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "top5_accuracy": top_k_accuracy_score(labels, probs, k=5, labels=np.arange(N_CLASSES))}

def save_atomic(obj, path):
    tmp = path.with_suffix(".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def fit(run_name, train_rows, val_rows, lr_backbone):
    """Train one run (resumable). With validation rows, keep the epoch with the best validation macro-F1;
    without, keep the last epoch. Returns the epoch history and a summary; the kept weights are in <run>_best.pt."""
    config = {**CONFIG, "run": run_name, "lr_backbone": lr_backbone, "train_photos": len(train_rows)}
    last, best_path = CKPT_DIR / f"{run_name}_last.pt", CKPT_DIR / f"{run_name}_best.pt"
    train_loader = make_loader(train_rows, train=True)
    val_loader = make_loader(val_rows, train=False) if val_rows is not None else None
    model = new_model()
    optimizer, scheduler = new_optimiser(model, lr_backbone, len(train_loader))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    start, history, best = 0, [], {"epoch": 0, "val": None}
    if last.exists():
        ckpt = torch.load(last, map_location=DEVICE, weights_only=False)
        if ckpt["config"] != config:
            raise ValueError(f"{last.name} was saved with different settings - delete it to start this run again")
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        start, history, best = ckpt["epoch"] + 1, ckpt["history"], ckpt["best"]
        print(f"{run_name}: resuming after epoch {start}" if start < EPOCHS else f"{run_name}: already finished")

    for epoch in range(start, EPOCHS):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
        row = {"epoch": epoch + 1, "train_loss": train_loss, "train_accuracy": train_acc}
        if val_loader is not None:
            val_probs, val_labels, val_loss = predict(model, val_loader)
            val = scores(val_probs, val_labels)
            row.update(val_loss=val_loss, **{f"val_{k}": v for k, v in val.items()})
        row["seconds"] = time.time() - t0
        history.append(row)
        keep = val_loader is None or best["val"] is None or val["macro_f1"] > best["val"]["macro_f1"]
        if keep:
            best = {"epoch": epoch + 1, "val": val if val_loader is not None else None}
            save_atomic({"model": model.state_dict(), "config": config, **best}, best_path)
        save_atomic({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                     "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch,
                     "history": history, "best": best, "config": config}, last)
        msg = f"{run_name} | epoch {epoch + 1:>2}/{EPOCHS} | train loss {train_loss:.3f} acc {train_acc:.3f}"
        if val_loader is not None:
            msg += (f" | val loss {val_loss:.3f} acc {val['accuracy']:.3f} macro-F1 {val['macro_f1']:.3f}"
                    f" top-5 {val['top5_accuracy']:.3f}")
        print(msg + f" | {row['seconds']:.0f}s" + ("  *" if keep and val_loader is not None else ""), flush=True)

    history = pd.DataFrame(history)
    history.to_csv(OUT / f"{run_name}_history.csv", index=False)
    del model, optimizer, scheduler
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return history, {"run": run_name, "lr_backbone": lr_backbone, "best_epoch": best["epoch"], "val": best["val"],
                     "train_minutes": round(history.seconds.sum() / 60, 1)}

## 7. Select the learning rate (train on 80%, score on validation)

One run per backbone learning rate in `LR_GRID`, each scored on the validation photos after every epoch.

In [ ]:
selection = {lr: fit(f"{BASE_NAME}_lr{lr:g}", train_df, val_df, lr) for lr in LR_GRID}

selection_table = pd.DataFrame([{"backbone lr": lr, "run": s["run"], "best epoch": s["best_epoch"],
                                 "val accuracy": s["val"]["accuracy"], "val macro-F1": s["val"]["macro_f1"],
                                 "val top-5": s["val"]["top5_accuracy"], "minutes": s["train_minutes"]}
                                for lr, (_, s) in selection.items()])
BEST_LR = max(selection, key=lambda lr: selection[lr][1]["val"]["macro_f1"])
print(f"Chosen backbone learning rate: {BEST_LR:g}")
if BEST_LR in (min(LR_GRID), max(LR_GRID)) and len(LR_GRID) > 1:
    print("Note: the best rate is at the edge of LR_GRID, so a rate outside the grid might do better.")
selection_table.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for lr, (hist, s) in selection.items():
    line, = axes[0].plot(hist.epoch, hist.val_loss, label=f"validation, lr {lr:g}")
    axes[0].plot(hist.epoch, hist.train_loss, ":", color=line.get_color(), label=f"train (smoothed labels), lr {lr:g}")
    line, = axes[1].plot(hist.epoch, hist.val_macro_f1, label=f"validation macro-F1, lr {lr:g}")
    axes[1].plot(hist.epoch, hist.train_accuracy, ":", color=line.get_color(), label=f"train accuracy, lr {lr:g}")
axes[0].set(title="Loss", xlabel="epoch", ylabel="cross-entropy")
axes[1].set(title="Validation macro-F1 and training accuracy", xlabel="epoch", ylim=(0, 1))
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
fig.suptitle(f"{BASE_NAME}: learning-rate selection", fontweight="bold")
plt.tight_layout()
fig.savefig(FIGURES / f"{BASE_NAME}_curves.png", dpi=150, bbox_inches="tight")
plt.show()

A widening gap between training accuracy and validation macro-F1 means the network is memorising the training photos (overfitting). Validation macro-F1 levelling off shows where more epochs stop helping.

## 8. Final model (retrain on all training photos)

Once the learning rate is chosen the validation photos are not needed any more, so the final model is trained again from the ImageNet weights on all 5,994 training photos (~25% more photos per species), and the last epoch is kept.

In [ ]:
FINAL_NAME = f"{BASE_NAME}_final"
final_history, final_summary = fit(FINAL_NAME, full_train_df, None, BEST_LR)
print(f"Final model: {FINAL_NAME} ({final_summary['train_minutes']} min)")

## 9. Final test evaluation (run once)

The final model is scored once on the 5,794 official test photos, then compared with the attribute-based models and every other ResNet-50 run whose results are in the output folder.

In [ ]:
model = new_model()
final_ckpt = torch.load(CKPT_DIR / f"{FINAL_NAME}_best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(final_ckpt["model"])
test_probs, test_labels, test_loss = predict(model, make_loader(test_df, train=False))
test = scores(test_probs, test_labels)

results = {"run": FINAL_NAME, "config": {**CONFIG, "lr_backbone": BEST_LR, "final_retrain": True},
           "selection": selection_table.to_dict("records"), "final_epoch": final_ckpt["epoch"],
           "test": test, "test_photos": int(len(test_labels)),
           "train_minutes_total": round(sum(s["train_minutes"] for _, s in selection.values())
                                        + final_summary["train_minutes"], 1)}
(OUT / f"{FINAL_NAME}_results.json").write_text(json.dumps(results, indent=2))

per_class = pd.DataFrame({"class_id": np.arange(1, N_CLASSES + 1), "species": species,
                          "f1": f1_score(test_labels, test_probs.argmax(1), average=None, labels=np.arange(N_CLASSES))})
per_class.to_csv(OUT / f"{FINAL_NAME}_per_class_f1.csv", index=False)
print(f"{FINAL_NAME} | test accuracy {test['accuracy']:.4f} | macro-F1 {test['macro_f1']:.4f} | "
      f"top-5 {test['top5_accuracy']:.4f}")

# Best attribute-based results on the same official test split (from the notebooks named), plus every ResNet run
# whose results file is in the output folder
rows = [("Random forest, tuned (randomForest.ipynb)", "312 attributes", 0.4563, 0.4349, 0.7206),
        ("RF + logistic regression ensemble (randomForestOptimisation.ipynb)", "attributes + aggregates",
         0.5104, 0.5059, 0.7656)]
for path in sorted(OUT.glob("resnet50_*_results.json")):
    r = json.loads(path.read_text())
    rows.append((r["run"], "photos", r["test"]["accuracy"], r["test"]["macro_f1"], r["test"]["top5_accuracy"]))
pd.DataFrame(rows, columns=["model", "input", "test accuracy", "test macro-F1", "test top-5"]).round(4)

## 10. Which species does it confuse?

The most frequent test mistakes, as (true species → predicted species).

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_labels, test_probs.argmax(1), labels=np.arange(N_CLASSES))
np.fill_diagonal(cm, 0)
top = np.dstack(np.unravel_index(np.argsort(cm, axis=None)[::-1][:10], cm.shape))[0]
print(pd.DataFrame([(species[t], species[p], cm[t, p], int((test_labels == t).sum())) for t, p in top],
                   columns=["true species", "predicted as", "mistakes", "test photos of true species"]).to_string())

## 11. Files written

All in `MyDrive/f20dl_resnet50/` on Colab (`results/resnet50/` locally). Copy the small ones into the repo's `results/resnet50/` for the report:
- `resnet50_standard_224px_final_results.json`: settings, learning-rate selection, test scores
- `resnet50_standard_224px_*_history.csv`: one row per epoch for every run
- `resnet50_standard_224px_final_per_class_f1.csv`: test F1 for each species
- `figures/resnet50_standard_224px_curves.png`: learning curves
- `05_image_preparation_executed.ipynb` (Colab only): the executed pipeline notebook with its checks

The `checkpoints/` folder and the `cub_processed_*.zip` caches are large and only needed on Colab.